In [ ]:
# Install necessary libraries if not already installed
%pip install pandas plotly folium streamlit-folium

import sqlite3
import pandas as pd
import json
import os
import plotly.express as px

# Define the data directory
DATA_DIR = "data"
DB_PATH = "db/brickview.db"

# Create folders if they don't exist
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("db", exist_ok=True)

In [3]:
def get_cleaned_data():
    # 1. Listings
    df_listings = pd.DataFrame(json.load(open(f"{DATA_DIR}/listings_final_expanded.json")))
    df_listings.rename(columns={"Listing_ID": "listing_id", "Sqft": "area_sqft", "Date_Listed": "date_listed"}, inplace=True, errors='ignore')
    
    # 2. Attributes (Converting Bools to Ints for SQLite)
    df_attr = pd.DataFrame(json.load(open(f"{DATA_DIR}/property_attributes_final_expanded.json")))
    for col in ["is_rented", "parking_available", "power_backup"]:
        df_attr[col] = df_attr[col].astype(bool).astype(int)

    # 3. Agents
    df_agents = pd.DataFrame(json.load(open(f"{DATA_DIR}/agents_cleaned.json")))
    df_agents.columns = [c.lower() for c in df_agents.columns] # Standardize to lowercase

    # 4. Sales
    df_sales = pd.read_csv(f"{DATA_DIR}/sales_cleaned.csv")
    df_sales.columns = [c.lower() for c in df_sales.columns]

    # 5. Buyers
    df_buyers = pd.DataFrame(json.load(open(f"{DATA_DIR}/buyers_cleaned.json")))
    df_buyers.rename(columns={"sale_id": "listing_id"}, inplace=True)
    df_buyers["loan_taken"] = df_buyers["loan_taken"].astype(bool).astype(int)

    return df_listings, df_attr, df_agents, df_sales, df_buyers

df_listings, df_attr, df_agents, df_sales, df_buyers = get_cleaned_data()
print("✅ Data cleaned and loaded into memory.")

✅ Data cleaned and loaded into memory.


In [4]:
conn = sqlite3.connect(DB_PATH)

# Push to SQL
df_listings.to_sql('listings', conn, if_exists='replace', index=False)
df_attr.to_sql('attributes', conn, if_exists='replace', index=False)
df_agents.to_sql('agents', conn, if_exists='replace', index=False)
df_sales.to_sql('sales', conn, if_exists='replace', index=False)
df_buyers.to_sql('buyers', conn, if_exists='replace', index=False)

print(f"✅ Database created at {DB_PATH}")
conn.close()

✅ Database created at db/brickview.db


In [1]:
import sys
!{sys.executable} -m pip install --upgrade nbformat

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.2-py3-none-any.whl (24 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [nbformat]


In [1]:
import sqlite3
import pandas as pd
import plotly.express as px

# Connect and Query
conn = sqlite3.connect('db/brickview.db')
query = """
SELECT 
    l.City, 
    ROUND(AVG(l.price), 2) as avg_price,
    ROUND(AVG(s.days_on_market), 1) as avg_days_to_sell
FROM listings l
LEFT JOIN sales s ON l.listing_id = s.listing_id
GROUP BY l.City
"""
city_analysis = pd.read_sql(query, conn)
conn.close()

# Create Visual
fig = px.bar(
    city_analysis, 
    x='City', 
    y='avg_price', 
    color='avg_days_to_sell',
    title="Real Estate Market: Price vs. Velocity by City"
)

# Use 'iframe' if 'notebook' still gives issues; it's more robust in VS Code
fig.show(renderer="iframe") 

# Display table below it
city_analysis

,City,avg_price,avg_days_to_sell
0,Chicago,2430677.90,64.3
1,Houston,2436811.00,58.5
2,Los Angeles,2442418.31,65.1
3,New York,2493319.74,60.8
4,Phoenix,2459961.76,59.7


In [2]:
import pandas as pd

# Reading the Listings data (JSON)
listings_df = pd.read_json("data/listings_final_expanded.json")

# Reading the Property Attributes data (JSON)
property_df = pd.read_json("data/property_attributes_final_expanded.json")

# Reading the Agents data (JSON)
agents_df = pd.read_json("data/agents_cleaned.json")

# Reading the Sales data (CSV)
sales_df = pd.read_csv("data/sales_cleaned.csv")

# Reading the Buyers data (JSON)
buyers_df = pd.read_json("data/buyers_cleaned.json")

# Optional: Verify the data loaded correctly
print("Dataframes loaded successfully!")

Dataframes loaded successfully!


In [3]:
import pandas as pd

listings_df = pd.read_json("data/listings_final_expanded.json")
display_columns = [
    'Listing_ID', 'City', 'Property_Type', 
    'Price', 'Sqft', 'Date_Listed', 
    'Agent_ID', 'Latitude', 'Longitude'
]
listings_df[display_columns].head(10)

,Listing_ID,City,Property_Type,Price,Sqft,Date_Listed,Agent_ID,Latitude,Longitude
0,L00001,New York,Apartment,1.655144e+06,2753.009121,2023-05-06,A0015,33.965208,-69.861589
1,L00002,Los Angeles,Apartment,1.519141e+06,4966.988193,2023-02-14,A0038,42.547892,-90.277860
2,L00003,Houston,Apartment,1.624890e+05,1267.003959,2023-04-22,A0015,28.732327,-115.952982
3,L00004,Phoenix,Apartment,1.277016e+06,2128.014429,2024-01-02,A0042,26.403938,-74.771490
4,L00005,Phoenix,Townhouse,5.622970e+05,4178.997421,2023-10-29,A0018,39.425252,-83.917878
5,L00006,New York,House,1.564104e+06,3961.992616,2023-06-24,A0018,25.487793,-68.757114
6,L00007,Los Angeles,House,1.701163e+06,3257.015062,2023-02-22,A0006,44.969112,-112.678494
7,L00008,Houston,Apartment,8.528340e+05,3316.993721,2023-11-06,A0017,29.354913,-114.364681
8,L00009,New York,Townhouse,1.224551e+06,1522.007780,2024-05-13,A0025,32.302569,-94.556982
9,L00010,New York,Condo,1.839387e+06,3462.022639,2023-10-23,A0013,35.370397,-108.119318


In [4]:
import pandas as pd

# 1. FIX: Tell pandas to show numbers with 2 decimal places instead of scientific notation
pd.options.display.float_format = '{:.2f}'.format

# 2. Read the files from your data folder
listings_df = pd.read_json("data/listings_final_expanded.json")
property_df = pd.read_json("data/property_attributes_final_expanded.json")
agents_df = pd.read_json("data/agents_cleaned.json")
sales_df = pd.read_csv("data/sales_cleaned.csv")
buyers_df = pd.read_json("data/buyers_cleaned.json")

# 3. Clean the columns (Rounding) - This makes the data ready for SQL
listings_df['Price'] = listings_df['Price'].round(2)
listings_df['Sqft'] = listings_df['Sqft'].round(2)

# 4. Define columns to display (Note: case sensitive matching your JSON)
display_cols = [
    'Listing_ID', 'City', 'Property_Type', 
    'Price', 'Sqft', 'Date_Listed', 
    'Agent_ID', 'Latitude', 'Longitude'
]

# 5. Display the table
print("Table: Property Listings (Cleaned)")
listings_df[display_cols].head(10)

Table: Property Listings (Cleaned)


,Listing_ID,City,Property_Type,Price,Sqft,Date_Listed,Agent_ID,Latitude,Longitude
0,L00001,New York,Apartment,1655144.01,2753.01,2023-05-06,A0015,33.97,-69.86
1,L00002,Los Angeles,Apartment,1519141.00,4966.99,2023-02-14,A0038,42.55,-90.28
2,L00003,Houston,Apartment,162489.00,1267.00,2023-04-22,A0015,28.73,-115.95
3,L00004,Phoenix,Apartment,1277015.98,2128.01,2024-01-02,A0042,26.40,-74.77
4,L00005,Phoenix,Townhouse,562297.01,4179.00,2023-10-29,A0018,39.43,-83.92
5,L00006,New York,House,1564104.01,3961.99,2023-06-24,A0018,25.49,-68.76
6,L00007,Los Angeles,House,1701163.00,3257.02,2023-02-22,A0006,44.97,-112.68
7,L00008,Houston,Apartment,852834.01,3316.99,2023-11-06,A0017,29.35,-114.36
8,L00009,New York,Townhouse,1224551.00,1522.01,2024-05-13,A0025,32.30,-94.56
9,L00010,New York,Condo,1839386.99,3462.02,2023-10-23,A0013,35.37,-108.12


SQL INSERTION


In [5]:
import pandas as pd
import sqlite3
import os

# Suppress scientific notation
pd.options.display.float_format = '{:.2f}'.format

# Load datasets
listings_df = pd.read_json("data/listings_final_expanded.json")
property_df = pd.read_json("data/property_attributes_final_expanded.json")
agents_df = pd.read_json("data/agents_cleaned.json")
sales_df = pd.read_csv("data/sales_cleaned.csv")
buyers_df = pd.read_json("data/buyers_cleaned.json")

# Inspecting data types for Schema creation
print("--- Listings Info ---")
print(listings_df.dtypes)
print("\n--- Sales Info ---")
print(sales_df.dtypes)

--- Listings Info ---
Listing_ID           str
City                 str
Property_Type        str
Price            float64
Sqft             float64
Date_Listed          str
Agent_ID             str
Latitude         float64
Longitude        float64
dtype: object

--- Sales Info ---
Listing_ID            str
Sale_Price        float64
Date_Sold             str
Days_on_Market    float64
dtype: object


SCHEMAS FOR ALL TABLES

In [6]:
# Connect to the database
conn = sqlite3.connect('db/brickview_manual.db')
cursor = conn.cursor()

# 1. Create Agents Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Agents (
    Agent_ID TEXT PRIMARY KEY,
    Name TEXT,
    Phone TEXT,
    Email TEXT,
    Commission_Rate REAL,
    Deals_Closed INTEGER,
    Rating REAL,
    Experience_Years INTEGER,
    Avg_Closing_Days INTEGER
)''')

# 2. Create Listings Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Listings (
    Listing_ID TEXT PRIMARY KEY,
    City TEXT,
    Property_Type TEXT,
    Price REAL,
    Sqft REAL,
    Date_Listed TEXT,
    Agent_ID TEXT,
    Latitude REAL,
    Longitude REAL,
    FOREIGN KEY (Agent_ID) REFERENCES Agents (Agent_ID)
)''')

# 3. Create Property_Attributes Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Property_Attributes (
    attribute_id INTEGER PRIMARY KEY,
    listing_id TEXT,
    bedrooms INTEGER,
    bathrooms INTEGER,
    floor_number INTEGER,
    total_floors INTEGER,
    year_built INTEGER,
    is_rented INTEGER,
    tenant_count INTEGER,
    furnishing_status TEXT,
    metro_distance_km REAL,
    parking_available INTEGER,
    power_backup INTEGER,
    FOREIGN KEY (listing_id) REFERENCES Listings (Listing_ID)
)''')

# 4. Create Sales Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Sales (
    Sale_ID INTEGER PRIMARY KEY AUTOINCREMENT,
    listing_id TEXT,
    sale_price REAL,
    date_sold TEXT,
    days_on_market REAL,
    FOREIGN KEY (listing_id) REFERENCES Listings (Listing_ID)
)''')

# 5. Create Buyers Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Buyers (
    buyer_id INTEGER PRIMARY KEY,
    listing_id TEXT,
    buyer_type TEXT,
    payment_mode TEXT,
    loan_taken INTEGER,
    loan_provider TEXT,
    loan_amount REAL,
    FOREIGN KEY (listing_id) REFERENCES Listings (Listing_ID)
)''')

conn.commit()
print("✅ Database Schema created successfully!")

✅ Database Schema created successfully!


MIGRATION

In [7]:
print("🚀 Starting Migration...")

# Helper function to migrate a dataframe to a table
def migrate_data(df, table_name, query):
    count = 0
    for index, row in df.iterrows():
        # Convert row to a tuple for SQL injection
        cursor.execute(query, tuple(row))
        count += 1
    conn.commit()
    print(f"✅ Migrated {count} rows to {table_name}")

# --- 1. Migrate Agents ---
agents_query = "INSERT INTO Agents VALUES (?,?,?,?,?,?,?,?,?)"
migrate_data(agents_df, "Agents", agents_query)

# --- 2. Migrate Listings ---
listings_query = "INSERT INTO Listings VALUES (?,?,?,?,?,?,?,?,?)"
migrate_data(listings_df, "Listings", listings_query)

# --- 3. Migrate Attributes ---
# Note: Ensure booleans are converted to ints (0/1) for SQLite
property_df['is_rented'] = property_df['is_rented'].astype(int)
property_df['parking_available'] = property_df['parking_available'].astype(int)
property_df['power_backup'] = property_df['power_backup'].astype(int)

attr_query = "INSERT INTO Property_Attributes VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)"
migrate_data(property_df, "Property_Attributes", attr_query)

# --- 4. Migrate Sales ---
sales_query = "INSERT INTO Sales (listing_id, sale_price, date_sold, days_on_market) VALUES (?,?,?,?)"
migrate_data(sales_df, "Sales", sales_query)

# --- 5. Migrate Buyers ---
# Rename column in memory to match schema if necessary, or just align order
buyers_df['loan_taken'] = buyers_df['loan_taken'].astype(int)
buyers_query = "INSERT INTO Buyers VALUES (?,?,?,?,?,?,?)"
migrate_data(buyers_df, "Buyers", buyers_query)

print("\n✨ Migration Complete!")
conn.close()

🚀 Starting Migration...
✅ Migrated 50 rows to Agents
✅ Migrated 21200 rows to Listings
✅ Migrated 21200 rows to Property_Attributes
✅ Migrated 720 rows to Sales
✅ Migrated 20000 rows to Buyers

✨ Migration Complete!


QUESTIONS AND QUERIES

1) PROPERTY AND PRICING ANALYSIS

In [8]:
# --- Question 1: Average listing price by city ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT City, ROUND(AVG(Price), 2) as Avg_Price 
FROM Listings 
GROUP BY City 
ORDER BY Avg_Price DESC
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

# Visualization
fig = px.bar(df_result, x='City', y='Avg_Price', title="Average Listing Price by City")
fig.show()

Insight Table:


,City,Avg_Price
0,New York,2493319.74
1,Phoenix,2459961.76
2,Los Angeles,2442418.31
3,Houston,2436811.00
4,Chicago,2430677.90


In [9]:
# --- Question 2: Average price per square foot by property type ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT Property_Type, ROUND(AVG(Price / Sqft), 2) as Avg_Price_Sqft 
FROM Listings 
GROUP BY Property_Type
ORDER BY Avg_Price_Sqft DESC
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,Property_Type,Avg_Price_Sqft
0,House,796.04
1,Apartment,792.20
2,Townhouse,789.75
3,Condo,754.68


In [10]:
# --- Question 3: Furnishing status impact on property prices ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT pa.furnishing_status, ROUND(AVG(l.Price), 2) as Avg_Price
FROM Listings l
JOIN Property_Attributes pa ON l.Listing_ID = pa.listing_id
GROUP BY pa.furnishing_status
ORDER BY Avg_Price DESC
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,furnishing_status,Avg_Price
0,Furnished,2463539.06
1,Semi-Furnished,2461192.65
2,Unfurnished,2433975.10


In [11]:
# --- Question 4: Metro proximity vs price ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT 
    CASE WHEN metro_distance_km <= 1 THEN '0-1km (Near)'
         WHEN metro_distance_km <= 3 THEN '1-3km (Moderate)'
         ELSE '3km+ (Far)' END as Metro_Proximity,
    ROUND(AVG(Price), 2) as Avg_Price
FROM Listings l
JOIN Property_Attributes pa ON l.Listing_ID = pa.listing_id
GROUP BY Metro_Proximity
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)# --- Question 4: Metro proximity vs price ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT 
    CASE WHEN metro_distance_km <= 1 THEN '0-1km (Near)'
         WHEN metro_distance_km <= 3 THEN '1-3km (Moderate)'
         ELSE '3km+ (Far)' END as Metro_Proximity,
    ROUND(AVG(Price), 2) as Avg_Price
FROM Listings l
JOIN Property_Attributes pa ON l.Listing_ID = pa.listing_id
GROUP BY Metro_Proximity
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,Metro_Proximity,Avg_Price
0,0-1km (Near),2386849.26
1,1-3km (Moderate),2399214.25
2,3km+ (Far),2467627.67


Insight Table:


,Metro_Proximity,Avg_Price
0,0-1km (Near),2386849.26
1,1-3km (Moderate),2399214.25
2,3km+ (Far),2467627.67


In [12]:
# --- Question 5: Rented vs Non-Rented Prices ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT 
    CASE WHEN is_rented = 1 THEN 'Rented' ELSE 'Not Rented' END as Status,
    ROUND(AVG(Price), 2) as Avg_Price
FROM Listings l
JOIN Property_Attributes pa ON l.Listing_ID = pa.listing_id
GROUP BY is_rented
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,Status,Avg_Price
0,Not Rented,2448250.30
1,Rented,2457184.90


In [13]:
# --- Question 6: Bedrooms/Bathrooms impact on price ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT bedrooms, bathrooms, ROUND(AVG(Price), 2) as Avg_Price
FROM Listings l
JOIN Property_Attributes pa ON l.Listing_ID = pa.listing_id
GROUP BY bedrooms, bathrooms
ORDER BY bedrooms, bathrooms
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,bedrooms,bathrooms,Avg_Price
0,1,1,2434001.41
1,1,2,2509474.77
2,1,3,2459032.59
3,1,4,2455532.96
4,2,1,2455735.36
5,2,2,2378687.66
6,2,3,2482618.93
7,2,4,2555766.23
8,3,1,2426007.97
9,3,2,2409122.73


In [14]:
# --- Question 7: Parking and Power Backup vs Price ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT 
    CASE WHEN parking_available = 1 THEN 'Yes' ELSE 'No' END as Parking,
    CASE WHEN power_backup = 1 THEN 'Yes' ELSE 'No' END as Backup,
    ROUND(AVG(Price), 2) as Avg_Price
FROM Listings l
JOIN Property_Attributes pa ON l.Listing_ID = pa.listing_id
GROUP BY parking_available, power_backup
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,Parking,Backup,Avg_Price
0,No,No,2436677.75
1,No,Yes,2446292.59
2,Yes,No,2462363.80
3,Yes,Yes,2465651.62


In [15]:
# --- Question 8: Influence of Year Built on Price ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT (year_built / 10 * 10) as Decade, ROUND(AVG(Price), 2) as Avg_Price
FROM Listings l
JOIN Property_Attributes pa ON l.Listing_ID = pa.listing_id
GROUP BY Decade
ORDER BY Decade
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,Decade,Avg_Price
0,1990,2507601.78
1,2000,2434914.69
2,2010,2408097.43
3,2020,2475700.08


In [16]:
# --- Question 9: Cities with highest median prices ---
conn = sqlite3.connect('db/brickview_manual.db')

# SQLite approx for Median
query = """
SELECT City, Price as Median_Price
FROM (
    SELECT City, Price, 
    ROW_NUMBER() OVER (PARTITION BY City ORDER BY Price) as RowNum,
    COUNT(*) OVER (PARTITION BY City) as TotalCount
    FROM Listings
) WHERE RowNum = (TotalCount + 1) / 2
ORDER BY Median_Price DESC
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,City,Median_Price
0,New York,2446518.63
1,Phoenix,2397597.97
2,Los Angeles,2380084.64
3,Chicago,2351084.47
4,Houston,2347582.06


In [17]:
# --- Question 10: Property distribution by price buckets ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT 
    CASE WHEN Price < 500000 THEN 'Budget (<500k)'
         WHEN Price BETWEEN 500000 AND 1500000 THEN 'Mid-Range (500k-1.5M)'
         ELSE 'Luxury (>1.5M)' END as Price_Bucket,
    COUNT(*) as Property_Count
FROM Listings
GROUP BY Price_Bucket
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,Price_Bucket,Property_Count
0,Budget (<500k),1867
1,Luxury (>1.5M),14538
2,Mid-Range (500k-1.5M),4795


2) Sales & Market Performance

In [18]:
# --- Question 11: Avg days on market by city ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT l.City, ROUND(AVG(s.days_on_market), 1) as Avg_DOM
FROM Listings l
JOIN Sales s ON l.Listing_ID = s.listing_id
GROUP BY l.City
ORDER BY Avg_DOM ASC
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,City,Avg_DOM
0,Houston,58.50
1,Phoenix,59.70
2,New York,60.80
3,Chicago,64.30
4,Los Angeles,65.10


In [19]:
# --- Question 12: Property types that sell fastest ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT l.Property_Type, ROUND(AVG(s.days_on_market), 1) as Avg_DOM
FROM Listings l
JOIN Sales s ON l.Listing_ID = s.listing_id
GROUP BY l.Property_Type
ORDER BY Avg_DOM ASC
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,Property_Type,Avg_DOM
0,House,58.30
1,Apartment,60.60
2,Townhouse,61.00
3,Condo,66.50


In [20]:
# --- Question 13: % Properties sold above list price ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT 
    ROUND(COUNT(CASE WHEN s.sale_price > l.Price THEN 1 END) * 100.0 / COUNT(*), 2) as Pct_Above_List
FROM Sales s
JOIN Listings l ON s.listing_id = l.Listing_ID
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,Pct_Above_List
0,49.31


In [21]:
# --- Question 14: Sale-to-list ratio by city ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT l.City, ROUND(AVG(s.sale_price / l.Price), 3) as Sale_List_Ratio
FROM Listings l
JOIN Sales s ON l.Listing_ID = s.listing_id
GROUP BY l.City
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,City,Sale_List_Ratio
0,Chicago,1.00
1,Houston,1.00
2,Los Angeles,1.00
3,New York,1.00
4,Phoenix,1.00


In [22]:
# --- Question 15: Listings taking > 90 days to sell ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT listing_id, ROUND(days_on_market, 0) as DOM
FROM Sales
WHERE days_on_market > 90
ORDER BY days_on_market DESC
"""

df_result = pd.read_sql(query, conn).head(10) # Top 10 for display
conn.close()

print("Insight Table (Top 10):")
display(df_result)

Insight Table (Top 10):


,listing_id,DOM
0,L00157,120.00
1,L00179,120.00
2,L00259,120.00
3,L00067,120.00
4,L00287,120.00
5,L00078,119.00
6,L01134,119.00
7,L00351,119.00
8,L00057,119.00
9,L00690,119.00


In [23]:
# --- Question 16: Metro distance vs Time on Market ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT 
    CASE WHEN pa.metro_distance_km < 2 THEN 'Near Metro (<2km)' 
         ELSE 'Far from Metro (>2km)' END as Location,
    ROUND(AVG(s.days_on_market), 1) as Avg_DOM
FROM Sales s
JOIN Property_Attributes pa ON s.listing_id = pa.listing_id
GROUP BY Location
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,Location,Avg_DOM
0,Far from Metro (>2km),60.80
1,Near Metro (<2km),64.70


In [24]:
# --- Question 17: Monthly sales trend ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT strftime('%Y-%m', date_sold) as Sale_Month, COUNT(*) as Total_Sales
FROM Sales
GROUP BY Sale_Month
ORDER BY Sale_Month
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

# Visualization
fig = px.line(df_result, x='Sale_Month', y='Total_Sales', title="Monthly Sales Trends")
fig.show()

Insight Table:


,Sale_Month,Total_Sales
0,2023-01,7
1,2023-02,17
2,2023-03,20
3,2023-04,36
4,2023-05,45
5,2023-06,31
6,2023-07,48
7,2023-08,50
8,2023-09,46
9,2023-10,64


In [25]:
# --- Question 18: Currently unsold properties ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT Listing_ID, City, Property_Type, Price
FROM Listings
WHERE Listing_ID NOT IN (SELECT listing_id FROM Sales)
LIMIT 10
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table (Sample of 10):")
display(df_result)

Insight Table (Sample of 10):


,Listing_ID,City,Property_Type,Price
0,L00002,Los Angeles,Apartment,1519141.00
1,L00005,Phoenix,Townhouse,562297.01
2,L00009,New York,Townhouse,1224551.00
3,L00014,Chicago,Apartment,1377440.01
4,L00015,Los Angeles,House,1069428.99
5,L00016,Phoenix,House,1535739.99
6,L00017,New York,House,1823443.99
7,L00018,Houston,Condo,238807.00
8,L00020,Houston,House,655493.00
9,L00021,Phoenix,Condo,1666600.99


In [26]:
# --- Question 19: Agents with most closed sales ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT a.Name, COUNT(s.listing_id) as Closed_Deals
FROM Agents a
JOIN Listings l ON a.Agent_ID = l.Agent_ID
JOIN Sales s ON l.Listing_ID = s.listing_id
GROUP BY a.Name
ORDER BY Closed_Deals DESC
LIMIT 10
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table (Top 10):")
display(df_result)

Insight Table (Top 10):


,Name,Closed_Deals
0,Agent A0042,25
1,Agent A0011,24
2,Agent A0035,21
3,Agent A0014,21
4,Agent A0046,20
5,Agent A0043,20
6,Agent A0048,19
7,Agent A0007,19
8,Agent A0029,18
9,Agent A0027,18


In [27]:
# --- Question 20: Top agents by total revenue ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT a.Name, ROUND(SUM(s.sale_price), 2) as Total_Revenue
FROM Agents a
JOIN Listings l ON a.Agent_ID = l.Agent_ID
JOIN Sales s ON l.Listing_ID = s.listing_id
GROUP BY a.Name
ORDER BY Total_Revenue DESC
LIMIT 10
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table (Top 10):")
display(df_result)

Insight Table (Top 10):


,Name,Total_Revenue
0,Agent A0011,27882272.02
1,Agent A0042,27191605.99
2,Agent A0043,24102418.02
3,Agent A0035,22725751.99
4,Agent A0014,22034008.02
5,Agent A0046,21335804.97
6,Agent A0048,21186306.00
7,Agent A0027,21099696.96
8,Agent A0009,20279274.97
9,Agent A0029,19586498.02


In [28]:
# --- Question 21: Agents who close deals fastest ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT a.Name, ROUND(AVG(s.days_on_market), 1) as Avg_Speed
FROM Agents a
JOIN Listings l ON a.Agent_ID = l.Agent_ID
JOIN Sales s ON l.Listing_ID = s.listing_id
GROUP BY a.Name
ORDER BY Avg_Speed ASC
LIMIT 10
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table (Top 10):")
display(df_result)

Insight Table (Top 10):


,Name,Avg_Speed
0,Agent A0044,36.90
1,Agent A0013,42.40
2,Agent A0037,44.50
3,Agent A0002,45.50
4,Agent A0008,46.60
5,Agent A0017,48.60
6,Agent A0026,49.50
7,Agent A0012,50.60
8,Agent A0010,51.10
9,Agent A0043,52.40


In [29]:
# --- Question 22: Experience vs Deals Closed ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT Experience_Years, ROUND(AVG(Deals_Closed), 1) as Avg_Deals
FROM Agents
GROUP BY Experience_Years
ORDER BY Experience_Years
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,Experience_Years,Avg_Deals
0,1,262.00
1,2,50.00
2,3,14.00
3,4,140.00
4,5,88.30
5,7,73.00
6,9,283.00
7,10,93.50
8,11,109.00
9,12,257.00


In [30]:
# --- Question 23: Rating vs Closing Speed ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT Rating, ROUND(AVG(avg_closing_days), 1) as Avg_Closing_Time
FROM Agents
GROUP BY Rating
ORDER BY Rating DESC
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,Rating,Avg_Closing_Time
0,5.00,53.50
1,4.90,54.50
2,4.80,51.00
3,4.60,36.00
4,4.50,61.00
5,4.40,38.00
6,4.30,58.80
7,4.20,26.00
8,4.10,55.80
9,4.00,27.00


In [31]:
# --- Question 24: Avg commission earned per agent ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT a.Name, ROUND(AVG(s.sale_price * (a.Commission_Rate / 100)), 2) as Avg_Comm
FROM Agents a
JOIN Listings l ON a.Agent_ID = l.Agent_ID
JOIN Sales s ON l.Listing_ID = s.listing_id
GROUP BY a.Name
ORDER BY Avg_Comm DESC
LIMIT 10
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table (Top 10):")
display(df_result)

Insight Table (Top 10):


,Name,Avg_Comm
0,Agent A0009,33997.61
1,Agent A0002,32326.16
2,Agent A0035,32032.49
3,Agent A0012,31737.61
4,Agent A0030,31665.09
5,Agent A0018,31438.36
6,Agent A0031,30358.05
7,Agent A0050,28564.40
8,Agent A0017,28406.74
9,Agent A0024,27805.90


In [32]:
# --- Question 25: Agents with most active listings ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT a.Name, COUNT(l.Listing_ID) as Active_Count
FROM Agents a
JOIN Listings l ON a.Agent_ID = l.Agent_ID
WHERE l.Listing_ID NOT IN (SELECT listing_id FROM Sales)
GROUP BY a.Name
ORDER BY Active_Count DESC
LIMIT 10
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table (Top 10):")
display(df_result)

Insight Table (Top 10):


,Name,Active_Count
0,Agent A0023,446
1,Agent A0011,439
2,Agent A0008,438
3,Agent A0042,435
4,Agent A0014,432
5,Agent A0044,430
6,Agent A0020,426
7,Agent A0048,425
8,Agent A0015,425
9,Agent A0012,425


3) Buyer & Financing Behavior

In [33]:
# --- Question 26: Investor vs End User % ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT buyer_type, COUNT(*) * 100.0 / (SELECT COUNT(*) FROM Buyers) as Percentage
FROM Buyers
GROUP BY buyer_type
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,buyer_type,Percentage
0,End User,50.10
1,Investor,49.90


In [34]:
# --- Question 27: Loan uptake rate by city ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT l.City, ROUND(AVG(b.loan_taken) * 100, 2) as Loan_Uptake_Pct
FROM Listings l
JOIN Buyers b ON l.Listing_ID = b.listing_id
GROUP BY l.City
ORDER BY Loan_Uptake_Pct DESC
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,City,Loan_Uptake_Pct
0,Los Angeles,50.99
1,Chicago,50.57
2,New York,50.22
3,Houston,49.71
4,Phoenix,49.50


In [35]:
# --- Question 28: Avg loan amount by buyer type ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT buyer_type, ROUND(AVG(loan_amount), 2) as Avg_Loan
FROM Buyers
WHERE loan_taken = 1
GROUP BY buyer_type
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,buyer_type,Avg_Loan
0,End User,5210307.45
1,Investor,5211545.97


In [36]:
# --- Question 29: Most common payment mode ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT payment_mode, COUNT(*) as Count
FROM Buyers
GROUP BY payment_mode
ORDER BY Count DESC
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,payment_mode,Count
0,Cash,5088
1,UPI,5012
2,Cheque,4951
3,Bank Transfer,4949


In [37]:
# --- Question 30: Loan impact on closing time ---
conn = sqlite3.connect('db/brickview_manual.db')

query = """
SELECT 
    CASE WHEN b.loan_taken = 1 THEN 'Loan' ELSE 'Cash/Other' END as Type,
    ROUND(AVG(s.days_on_market), 1) as Avg_DOM
FROM Buyers b
JOIN Sales s ON b.listing_id = s.listing_id
GROUP BY Type
"""

df_result = pd.read_sql(query, conn)
conn.close()

print("Insight Table:")
display(df_result)

Insight Table:


,Type,Avg_DOM
0,Cash/Other,61.10
1,Loan,62.10
